In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [4]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 37.3 MB/s eta 0:00:0000:0100:01m


In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

# 1. Configuración

In [6]:
# Rutas estándar en Kaggle
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')
TEST_PATH = os.path.join(DATA_DIR, 'test.parquet')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

# 2. Carga y Exploración de Datos (EDA)

In [7]:
print("Cargando datasets...")
train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("\n--- RESUMEN DEL DATASET DE ENTRENAMIENTO ---")
print(f"Total de espectros en Train: {len(train_df):,}")
print(f"Estructuras químicas únicas (inchikey14): {train_df['inchikey14'].nunique():,}")
print(f"Número de librerías fuente: {train_df['ingest_lib'].nunique()}")

print("\n--- RESUMEN DEL DATASET DE TEST ---")
print(f"Total de espectros en Test: {len(test_df):,}")
print(f"Moléculas únicas en Test (molecule_id): {test_df['molecule_id'].nunique():,}")

# Distribución de adductos en Test
print("\nTop adductos en Test:")
print(test_df['adduct'].value_counts())

Cargando datasets...

--- RESUMEN DEL DATASET DE ENTRENAMIENTO ---
Total de espectros en Train: 2,539,608
Estructuras químicas únicas (inchikey14): 275,810
Número de librerías fuente: 11

--- RESUMEN DEL DATASET DE TEST ---
Total de espectros en Test: 1,213
Moléculas únicas en Test (molecule_id): 400

Top adductos en Test:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64


# 3. Implementación de la métrica oficial MRR@25

In [8]:
def smiles_to_inchikey14(smiles: str) -> str:
    """
    Convierte un SMILES a su InChIKey14 aplicando la canonicalización de
    tautómeros estandarizada por RDKit según las reglas del concurso.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return ""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        
        # Canonicalización de tautómeros de RDKit
        enumerator = rdMolStandardize.TautomerEnumerator()
        canon_mol = enumerator.Canonicalize(mol)
        
        inchikey = Chem.MolToInchiKey(canon_mol)
        if inchikey:
            return inchikey.split('-')[0]  # Primer bloque de 14 caracteres
    except Exception:
        return ""
    return ""

def calculate_mrr25(y_true_inchikey14: dict, y_pred_smiles: dict) -> float:
    """
    Calcula el Mean Reciprocal Rank @ 25 (MRR@25).
    
    y_true_inchikey14: dict {molecule_id: inchikey14_target}
    y_pred_smiles: dict {molecule_id: [lista de hasta 25 SMILES]}
    """
    ranks = []
    
    for mol_id, target_key in y_true_inchikey14.items():
        preds = y_pred_smiles.get(mol_id, [])[:25]
        score = 0.0
        
        for rank_idx, pred_smiles in enumerate(preds, start=1):
            pred_key = smiles_to_inchikey14(pred_smiles)
            if pred_key != "" and pred_key == target_key:
                score = 1.0 / rank_idx
                break  # Detenerse en la primera coincidencia correcta
                
        ranks.append(score)
        
    return float(np.mean(ranks))

# Sanity Check de la métrica con un par de SMILES equivalente
smiles_1 = "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O"
smiles_2 = "OCC1OC(O)C(O)C(O)C1O"

key1 = smiles_to_inchikey14(smiles_1)
key2 = smiles_to_inchikey14(smiles_2)

print(f"InChIKey14 con estereoquímica: {key1}")
print(f"InChIKey14 plano/canónico:   {key2}")
print(f"¿Coinciden correctamente?:    {key1 == key2}")

InChIKey14 con estereoquímica: WQZGKKKJIJFFOK
InChIKey14 plano/canónico:   WQZGKKKJIJFFOK
¿Coinciden correctamente?:    True


# 4. Creación del Esquema de Validación (GroupKFold)

In [10]:
train_df.columns

Index(['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14',
       'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct',
       'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs',
       'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity',
       'collision_energy_ev', 'collision_energy_orig',
       'collision_energy_orig_units', 'fold'],
      dtype='object')

In [11]:
# Asignamos 5 folds agrupando estrictamente por la estructura 2D (inchikey14)
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['inchikey14'])):
    train_df.loc[val_idx, 'fold'] = fold

print("Distribución de espectros por fold:")
print(train_df['fold'].value_counts())

cols_to_save = [
    'normalized_smiles', 
    'inchikey', 
    'inchikey14', 
    'molecular_formula', 
    'precursor_mz', 
    'fold'
]
train_df[cols_to_save].to_parquet('train_folds.parquet')

print("\n✓ Se guardó 'train_folds.parquet' correctamente.")

Distribución de espectros por fold:
fold
0    507922
1    507922
2    507922
3    507921
4    507921
Name: count, dtype: int64

✓ Se guardó 'train_folds.parquet' correctamente.


In [ ]:
/kaggle/working/train_folds.parquet